# LeetCode 3312 — Sorted GCD Pair Queries bằng Python

- Link: https://leetcode.com/problems/sorted-gcd-pair-queries/
- Độ khó: **Hard**
- Chủ đề: **Đếm cặp theo gcd đúng bằng `g`** bằng đếm bội + **trừ dần từ lớn xuống nhỏ** (hoặc Möbius), **tổng tiền tố** + **tìm kiếm nhị phân** cho truy vấn phần tử thứ `q`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def gcdValues(self, nums: List[int], queries: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Không liệt kê `n(n−1)/2` cặp (tới `5·10^9`) — thay vào đó đếm `exact[g]` = số cặp có gcd **đúng bằng** `g`, với `g ≤ max(nums) ≤ 5·10^4`.
2. Đếm "gcd là **bội** của `g`" rất dễ: `C(c_g, 2)` với `c_g` = số phần tử chia hết cho `g`.
3. Suy ra `exact[g] = C(c_g, 2) − Σ_{k ≥ 2} exact[k·g]` bằng cách duyệt `g` **giảm dần** — tổng `O(M log M)` (chuỗi điều hòa).
4. Mảng gcd đã sắp chính là các giá trị `g` lặp `exact[g]` lần ⇒ truy vấn chỉ số `q` = tìm `g` nhỏ nhất có tổng tiền tố `> q` (bisect).
5. Kiểm chứng bằng vét cạn mọi cặp.

## 2. Đề bài & ví dụ

Cho `nums` (độ dài `n`) và `queries`. Gọi `gcdPairs` là mảng gcd của mọi cặp `(nums[i], nums[j])` với `i < j`, **sắp tăng dần**. Với mỗi truy vấn `q`, trả về `gcdPairs[q]` (chỉ số từ 0).

**Ví dụ 1:** `nums = [2,3,4]`, `queries = [0,2,2]` → `[1,2,2]` (`gcdPairs = [1,1,2]`).
**Ví dụ 2:** `nums = [4,4,2,1]`, `queries = [5,3,1,0]` → `[4,2,1,1]`.
**Ví dụ 3:** `nums = [2,2]`, `queries = [0,0]` → `[2,2]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 5·10^4`; `1 ≤ len(queries) ≤ 10^5`; `0 ≤ queries[i] < n(n−1)/2`.

In [1]:
from typing import List
from math import gcd, comb
import bisect

examples = [([2, 3, 4], [0, 2, 2], [1, 2, 2]), ([4, 4, 2, 1], [5, 3, 1, 0], [4, 2, 1, 1]), ([2, 2], [0, 0], [2, 2])]
for nums, qs, exp in examples:
    print(nums, qs, '-> kỳ vọng', exp)

[2, 3, 4] [0, 2, 2] -> kỳ vọng [1, 2, 2]
[4, 4, 2, 1] [5, 3, 1, 0] -> kỳ vọng [4, 2, 1, 1]
[2, 2] [0, 0] -> kỳ vọng [2, 2]


## 3. Kiến thức nền

### 3.1. Đếm số phần tử chia hết cho `g`

`freq[v]` = số lần `v` xuất hiện; `c_g = Σ_{v bội của g} freq[v]` — duyệt bội: tổng `Σ_g M/g = O(M log M)`.

In [2]:
def multiples_count(nums):
    M = max(nums)
    freq = [0] * (M + 1)
    for v in nums:
        freq[v] += 1
    c = [0] * (M + 1)
    for g in range(1, M + 1):
        c[g] = sum(freq[g::g])
    return c

print('c_g của [4,4,2,1]:', multiples_count([4, 4, 2, 1])[1:])

c_g của [4,4,2,1]: [4, 3, 0, 2]


### 3.2. Từ "bội" sang "đúng bằng"

Số cặp có gcd là bội của `g` = `C(c_g, 2)` (cả hai phần tử chia hết cho `g` ⇔ `g | gcd`). Các cặp này có gcd ∈ `{g, 2g, 3g, …}` ⇒ `exact[g] = C(c_g, 2) − Σ_{k≥2} exact[kg]`. Duyệt `g` từ lớn xuống nhỏ thì vế phải đã biết.

### 3.3. Truy vấn phần tử thứ `q` của mảng đã sắp

`pre[g] = Σ_{h ≤ g} exact[h]`. Phần tử ở chỉ số `q` là `g` nhỏ nhất với `pre[g] > q` ⇒ `bisect_right(pre, q)`.

In [3]:
pre = [0, 3, 5, 5, 6]        # ví dụ 2: exact[1]=3, exact[2]=2, exact[4]=1  (pre[0] = 0)
print([bisect.bisect_right(pre, q) for q in [5, 3, 1, 0]])

[4, 2, 1, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `nums = [4, 4, 2, 1]`, `M = 4`.

| `g` | `c_g` | `C(c_g, 2)` | trừ bội | `exact[g]` |
|---|---|---|---|---|
| 4 | 2 | 1 | — | 1 |
| 3 | 0 | 0 | — | 0 |
| 2 | 3 | 3 | `exact[4] = 1` | 2 |
| 1 | 4 | 6 | `exact[2] + exact[3] + exact[4] = 3` | 3 |

⇒ `gcdPairs = [1, 1, 1, 2, 2, 4]` ✓.

## 5. Cách 1 — Brute Force: mọi cặp

In [4]:
def brute(nums, queries):
    pairs = sorted(gcd(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))
    return [pairs[q] for q in queries]

for nums, qs, exp in examples:
    print(brute(nums, qs), '| kỳ vọng', exp)

[1, 2, 2] | kỳ vọng [1, 2, 2]
[4, 2, 1, 1] | kỳ vọng [4, 2, 1, 1]
[2, 2] | kỳ vọng [2, 2]


**Độ phức tạp:** `O(n² log n)` — chỉ cho `n` nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. `c_g` cho mọi `g ≤ M` (`O(M log M)`).
2. `exact[g]` duyệt `g = M..1`: `exact[g] = C(c_g, 2) − Σ_{k ≥ 2, kg ≤ M} exact[kg]` (`O(M log M)`).
3. `pre` tổng tiền tố; mỗi truy vấn `bisect_right(pre, q)` (`O(log M)`).

Tổng `O(n + M log M + Q log M)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đếm theo gcd bằng từ điển, cặp theo **giá trị phân biệt**

Gom `nums` theo giá trị (`d` giá trị phân biệt). Cặp hai giá trị khác nhau `u < v`: `freq[u]·freq[v]` cặp có gcd `gcd(u, v)`; cặp cùng giá trị `v`: `C(freq[v], 2)` cặp có gcd `v`. `O(d²)` — tốt khi ít giá trị phân biệt.

In [5]:
from collections import Counter
def distinct_pairs(nums, queries):
    fr = Counter(nums)
    vals = sorted(fr)
    cnt = Counter()
    for i, u in enumerate(vals):
        cnt[u] += comb(fr[u], 2)
        for v in vals[i + 1:]:
            cnt[gcd(u, v)] += fr[u] * fr[v]
    keys = sorted(cnt)
    pre, s = [], 0
    for k in keys:
        s += cnt[k]; pre.append(s)
    return [keys[bisect.bisect_right(pre, q)] for q in queries]

for nums, qs, exp in examples:
    print(distinct_pairs(nums, qs), '| kỳ vọng', exp)

[1, 2, 2] | kỳ vọng [1, 2, 2]
[4, 2, 1, 1] | kỳ vọng [4, 2, 1, 1]
[2, 2] | kỳ vọng [2, 2]


### 7.2. Cách 3 — Trừ dần bội số từ lớn xuống nhỏ (lời giải chính)

In [6]:
def subtract_multiples(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        e = c[g] * (c[g] - 1) // 2
        for kg in range(2 * g, M + 1, g):
            e -= exact[kg]
        exact[g] = e
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_right(pre, q) for q in queries]

for nums, qs, exp in examples:
    print(subtract_multiples(nums, qs), '| kỳ vọng', exp)

[1, 2, 2] | kỳ vọng [1, 2, 2]
[4, 2, 1, 1] | kỳ vọng [4, 2, 1, 1]
[2, 2] | kỳ vọng [2, 2]


### 7.3. Cách 4 — Nghịch đảo Möbius

`exact[g] = Σ_{k ≥ 1} μ(k) · C(c_{kg}, 2)` — cùng độ phức tạp, không phụ thuộc thứ tự duyệt.

In [7]:
def mobius_table(M):
    mu = [1] * (M + 1); is_p = [True] * (M + 1)
    for p in range(2, M + 1):
        if is_p[p]:
            for q in range(2 * p, M + 1, p):
                is_p[q] = False
            for q in range(p, M + 1, p):
                mu[q] = -mu[q]
            for q in range(p * p, M + 1, p * p):
                mu[q] = 0
    return mu

def mobius_solution(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    mu = mobius_table(M)
    exact = [0] * (M + 1)
    for g in range(1, M + 1):
        exact[g] = sum(mu[k] * (c[k * g] * (c[k * g] - 1) // 2) for k in range(1, M // g + 1) if mu[k])
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_right(pre, q) for q in queries]

for nums, qs, exp in examples:
    print(mobius_solution(nums, qs), '| kỳ vọng', exp)

[1, 2, 2] | kỳ vọng [1, 2, 2]
[4, 2, 1, 1] | kỳ vọng [4, 2, 1, 1]
[2, 2] | kỳ vọng [2, 2]


## 8. TRACE từng bước

Trace cách 3 trên `nums = [6, 4, 9, 12, 3]`.

In [8]:
def trace(nums):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        e = c[g] * (c[g] - 1) // 2
        sub = [exact[kg] for kg in range(2 * g, M + 1, g)]
        exact[g] = e - sum(sub)
        if c[g] >= 2:
            print(f"g = {g:2d}: c = {c[g]}, C(c,2) = {e}, trừ {sum(sub)} -> exact = {exact[g]}")
    pairs = sorted(gcd(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))
    print('gcdPairs (vét cạn):', pairs)
    print('từ exact          :', [g for g in range(1, M + 1) for _ in range(exact[g])])

trace([6, 4, 9, 12, 3])

g =  6: c = 2, C(c,2) = 1, trừ 0 -> exact = 1
g =  4: c = 2, C(c,2) = 1, trừ 0 -> exact = 1
g =  3: c = 4, C(c,2) = 6, trừ 1 -> exact = 5
g =  2: c = 3, C(c,2) = 3, trừ 2 -> exact = 1
g =  1: c = 5, C(c,2) = 10, trừ 8 -> exact = 2
gcdPairs (vét cạn): [1, 1, 2, 3, 3, 3, 3, 3, 4, 6]
từ exact          : [1, 1, 2, 3, 3, 3, 3, 3, 4, 6]


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def gcdValues(self, nums: List[int], queries: List[int]) -> List[int]:
        M = max(nums)
        freq = [0] * (M + 1)
        for v in nums:
            freq[v] += 1
        exact = [0] * (M + 1)
        for g in range(M, 0, -1):
            c = sum(freq[g::g])
            e = c * (c - 1) // 2
            for kg in range(2 * g, M + 1, g):
                e -= exact[kg]
            exact[g] = e
        pre = [0] * (M + 1)
        for g in range(1, M + 1):
            pre[g] = pre[g - 1] + exact[g]
        return [bisect.bisect_right(pre, q) for q in queries]

for nums, qs, exp in examples:
    got = Solution().gcdValues(nums, qs)
    print(got, 'OK' if got == exp else 'SAI')

[1, 2, 2] OK
[4, 2, 1, 1] OK
[2, 2] OK


`sum(freq[g::g])` dùng slicing (chạy bằng C) — nhanh hơn vòng lặp Python thuần.

## 10. Vì sao thuật toán đúng

1. **Đếm bội:** cặp `(i, j)` có `g | gcd(nums[i], nums[j])` ⇔ `g | nums[i]` và `g | nums[j]` ⇒ có `C(c_g, 2)` cặp.
2. **Phân hoạch:** các cặp đó chia theo giá trị gcd thật `h ∈ {g, 2g, …}` ⇒ `C(c_g, 2) = Σ_{k ≥ 1} exact[kg]`. Duyệt giảm dần giải được `exact[g]` duy nhất.
3. **Truy vấn:** mảng sắp gồm `exact[1]` số 1, rồi `exact[2]` số 2, … Phần tử chỉ số `q` là `g` nhỏ nhất mà số phần tử `≤ g` vượt `q`, tức `pre[g] > q` — đúng định nghĩa `bisect_right(pre, q)` (vì `pre` không giảm, `pre[0] = 0`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n² log n)` | `O(n²)` |
| 2. Cặp giá trị phân biệt | `O(d² log M)` | `O(M)` |
| 3. Trừ dần bội | `O(n + M log M + Q log M)` | `O(M)` |
| 4. Möbius | `O(n + M log M + Q log M)` | `O(M)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'distinct_pairs': distinct_pairs, 'subtract': subtract_multiples, 'mobius': mobius_solution,
         'submit': lambda a, q: Solution().gcdValues(a, q)}
edge = [([1, 1], [0], [1]), ([7, 14], [0], [7]), ([5, 10, 15], [0, 1, 2], [5, 5, 5]),
        ([6, 10, 15], [0, 1, 2], [2, 3, 5]), ([50000, 50000, 25000], [0, 2], [25000, 50000])]
for nums, qs, exp in examples + edge:
    assert brute(nums, qs) == exp, (nums, brute(nums, qs))
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3312)
cnt = 0
for _ in range(400):
    n = rng.randint(2, 25)
    nums = [rng.randint(1, rng.choice([10, 60, 500])) for _ in range(n)]
    total = n * (n - 1) // 2
    qs = [rng.randrange(total) for _ in range(rng.randint(1, 10))]
    exp = brute(nums, qs)
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 400 bộ test.


### 12.3. Ca lớn `n = Q = 10^5`, giá trị tới `5·10^4`

In [12]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': [rng.randint(1, 5 * 10 ** 4) for _ in range(N)],
    'toàn 5e4': [5 * 10 ** 4] * N,
    'bội của 12': [12 * rng.randint(1, 4166) for _ in range(N)],
}
for name, nums in big.items():
    total = N * (N - 1) // 2
    qs = [rng.randrange(total) for _ in range(N)]
    t = time.time(); a = Solution().gcdValues(nums, qs); t1 = time.time() - t
    t = time.time(); b = mobius_solution(nums, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:12s}: min/max đáp án {min(a)}/{max(a)} | nộp {t1:.2f}s | Möbius {t2:.2f}s")
assert set(Solution().gcdValues(big['toàn 5e4'], [0, N * (N - 1) // 2 - 1])) == {5 * 10 ** 4}
assert min(Solution().gcdValues(big['bội của 12'], list(range(100)))) >= 12
print('toàn 5e4 -> mọi gcd = 5e4; bội của 12 -> mọi gcd >= 12: đúng.')

ngẫu nhiên  : min/max đáp án 1/18989 | nộp 0.06s | Möbius 0.11s
toàn 5e4    : min/max đáp án 50000/50000 | nộp 0.05s | Möbius 0.11s


bội của 12  : min/max đáp án 12/49668 | nộp 0.07s | Möbius 0.14s
toàn 5e4 -> mọi gcd = 5e4; bội của 12 -> mọi gcd >= 12: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng thẳng `C(c_g, 2)` làm số cặp có gcd **đúng** bằng `g`

In [13]:
def wrong_no_subtract(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + c[g] * (c[g] - 1) // 2          # SAI: đếm trùng các cặp gcd = 2g, 3g, ...
    return [bisect.bisect_right(pre, q) for q in queries]

print('sai :', wrong_no_subtract([4, 4, 2, 1], [5, 3, 1, 0]), '| đúng:', Solution().gcdValues([4, 4, 2, 1], [5, 3, 1, 0]))

sai : [1, 1, 1, 1] | đúng: [4, 2, 1, 1]


**Vì sao sai:** cặp `(4, 4)` được đếm ở cả `g = 1, 2, 4`. Tổng số "cặp" vượt `n(n−1)/2`.

### Lỗi 2 — Đếm cặp **có thứ tự** (`c_g²` hoặc `c_g(c_g − 1)`)

In [14]:
def wrong_ordered(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        exact[g] = c[g] * (c[g] - 1) - sum(exact[kg] for kg in range(2 * g, M + 1, g))   # SAI: gấp đôi
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_right(pre, q) for q in queries]

print('sai :', wrong_ordered([2, 3, 4], [0, 2, 2]), '| đúng:', Solution().gcdValues([2, 3, 4], [0, 2, 2]))

sai : [1, 1, 1] | đúng: [1, 2, 2]


**Vì sao sai:** đề chỉ lấy `i < j`; mỗi giá trị gcd bị nhân đôi số lần xuất hiện ⇒ vị trí `q` trỏ sai.

### Lỗi 3 — Dùng `bisect_left` thay vì `bisect_right`

In [15]:
def wrong_bisect_left(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        exact[g] = c[g] * (c[g] - 1) // 2 - sum(exact[kg] for kg in range(2 * g, M + 1, g))
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_left(pre, q) for q in queries]                       # SAI

print('sai :', wrong_bisect_left([4, 4, 2, 1], [5, 3, 1, 0]), '| đúng:', Solution().gcdValues([4, 4, 2, 1], [5, 3, 1, 0]))

sai : [2, 1, 1, 0] | đúng: [4, 2, 1, 1]


**Vì sao sai:** cần `pre[g] > q` (đã có **hơn** `q` phần tử `≤ g`, vì chỉ số từ 0). `bisect_left` tìm `pre[g] ≥ q` — lệch một (và `q = 0` trả về `g = 0`).

### Lỗi 4 — Duyệt `g` **tăng dần** khi trừ bội

In [16]:
def wrong_increasing(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(1, M + 1):                                                  # SAI: exact[kg] chưa tính
        exact[g] = c[g] * (c[g] - 1) // 2 - sum(exact[kg] for kg in range(2 * g, M + 1, g))
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_right(pre, q) for q in queries]

print('sai :', wrong_increasing([4, 4, 2, 1], [5, 3, 1, 0]), '| đúng:', Solution().gcdValues([4, 4, 2, 1], [5, 3, 1, 0]))

sai : [1, 1, 1, 1] | đúng: [4, 2, 1, 1]


**Vì sao sai:** khi tính `exact[1]` cần `exact[2], exact[3], …` đã đúng — chúng chỉ có khi duyệt từ lớn xuống.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cặp theo gcd" với giá trị nhỏ (`≤ 10^5`) ⇒ đếm bội + trừ dần / Möbius, độ phức tạp chuỗi điều hòa.
- "Phần tử thứ `q` của một multiset lớn đã sắp" ⇒ lưu histogram theo giá trị + tổng tiền tố + bisect, không bao giờ dựng mảng.
- `bisect_right(pre, q)` với `pre[0] = 0` là công thức chuẩn cho chỉ số từ 0.

## 15. Biến thể của bài

1. Đếm cặp có gcd **đúng bằng** `k` cho trước — chỉ cần `exact[k]`.
2. Tổng gcd của mọi cặp ⇒ `Σ g · exact[g]`.
3. lcm thay gcd — khó hơn nhiều (lcm có thể rất lớn).
4. Truy vấn trên **mảng con** `[l, r]` — cần cấu trúc khác (offline + Möbius cập nhật).

In [17]:
# Biến thể 1 và 2
def exact_table(nums):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        exact[g] = c[g] * (c[g] - 1) // 2 - sum(exact[kg] for kg in range(2 * g, M + 1, g))
    return exact

def sum_of_pair_gcds(nums):
    ex = exact_table(nums)
    return sum(g * e for g, e in enumerate(ex))

rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(1, 60) for _ in range(rng.randint(2, 20))]
    pairs = [gcd(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums))]
    ex = exact_table(nums)
    k = rng.randint(1, 60)
    assert (ex[k] if k < len(ex) else 0) == pairs.count(k)
    assert sum_of_pair_gcds(nums) == sum(pairs)
print('Biến thể 1, 2 khớp vét cạn.')

Biến thể 1, 2 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cặp | `O(n² log n)` | Oracle |
| 2 | Cặp giá trị phân biệt | `O(d²)` | Ít giá trị phân biệt |
| 3 | Trừ dần bội | `O(M log M)` | **Nộp bài** |
| 4 | Möbius | `O(M log M)` | Công thức đóng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [3, 6, 9]`, `queries = [0, 1, 2]`.

**Bài 2 (tính tay):** `nums = [2, 4, 8, 3]`, `queries = [5]`.

**Bài 3 (điền TODO):** trả về **số lần xuất hiện** của gcd ở mỗi truy vấn (tức `exact[gcdPairs[q]]`).

**Bài 4 (tự viết lại):** gcd của **bộ ba** `i < j < k` thay vì cặp.

**Bài 5 (mini challenge):** truy vấn "phần tử thứ `q` từ **lớn** xuống".

In [18]:
def multiplicity_todo(nums, queries):
    # TODO
    raise NotImplementedError

def triple_gcd_todo(nums, queries):
    # TODO
    raise NotImplementedError

def descending_todo(nums, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) gcd cặp: `3, 3, 3`. (2) Sắp: `[3, 3, 3]`. (3) `[3, 3, 3]`.

**Bài 2.** (1) Cặp: `(2,4)=2, (2,8)=2, (2,3)=1, (4,8)=4, (4,3)=1, (8,3)=1`. (2) Sắp `[1,1,1,2,2,4]`. (3) `[4]`.

**Bài 3.** (1) Tính `exact`. (2) Truy vấn như thường. (3) Trả `exact[g]`.

**Bài 4.** (1) `C(c_g, 3)` thay `C(c_g, 2)`. (2) Trừ dần như cũ. (3) Tổng tiền tố + bisect.

**Bài 5.** (1) Chỉ số từ lớn `q` ⇔ chỉ số từ nhỏ `T − 1 − q`. (2) `T = n(n−1)/2`. (3) Dùng lại lời giải.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().gcdValues([3, 6, 9], [0, 1, 2]) == [3, 3, 3]
assert Solution().gcdValues([2, 4, 8, 3], [5]) == [4] == brute([2, 4, 8, 3], [5])

def multiplicity(nums, queries):
    ex = exact_table(nums)
    return [ex[g] for g in Solution().gcdValues(nums, queries)]

assert multiplicity([4, 4, 2, 1], [5, 3, 1, 0]) == [1, 2, 3, 3]
print('Bài 1: [3, 3, 3] | Bài 2: [4] | Bài 3:', multiplicity([4, 4, 2, 1], [5, 3, 1, 0]))

Bài 1: [3, 3, 3] | Bài 2: [4] | Bài 3: [1, 2, 3, 3]


In [20]:
# Bài 4, 5
def triple_gcd(nums, queries):
    M = max(nums)
    c = multiples_count(nums)
    exact = [0] * (M + 1)
    for g in range(M, 0, -1):
        exact[g] = comb(c[g], 3) - sum(exact[kg] for kg in range(2 * g, M + 1, g))
    pre = [0] * (M + 1)
    for g in range(1, M + 1):
        pre[g] = pre[g - 1] + exact[g]
    return [bisect.bisect_right(pre, q) for q in queries]

def descending(nums, queries):
    T = len(nums) * (len(nums) - 1) // 2
    return Solution().gcdValues(nums, [T - 1 - q for q in queries])

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(3, 12); nums = [rng.randint(1, 40) for _ in range(n)]
    trip = sorted(gcd(gcd(nums[i], nums[j]), nums[k]) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n))
    qs = [rng.randrange(len(trip)) for _ in range(5)]
    assert triple_gcd(nums, qs) == [trip[q] for q in qs]
    pairs = sorted((gcd(nums[i], nums[j]) for i in range(n) for j in range(i + 1, n)), reverse=True)
    qs2 = [rng.randrange(len(pairs)) for _ in range(5)]
    assert descending(nums, qs2) == [pairs[q] for q in qs2]
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
freq theo giá trị; for g = M..1: c = sum(freq[g::g]); exact[g] = C(c,2) - Σ exact[kg]
pre[g] = pre[g-1] + exact[g]
answer = [bisect_right(pre, q) for q in queries]
```

**Checklist:**

- [ ] Trừ các bội để có gcd **đúng bằng** `g`?
- [ ] Duyệt `g` **giảm dần**?
- [ ] Cặp không thứ tự `C(c, 2)`?
- [ ] `bisect_right` với `pre[0] = 0` (chỉ số từ 0)?
- [ ] Không dựng mảng `gcdPairs` (tới `5·10^9` phần tử)?